# Portfolio Backtesting System - Database Setup

This notebook will:
- Create the `portfolio_backtesting` database
- Create all 9 tables
- Import all data (ETFs, benchmarks, price history)

**Prerequisites:**
- MySQL server running on localhost:3306
- Username: root, Password: krittanut123456
- Data files in `data/` folder

## 1. Import Libraries

In [ ]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
import os
from datetime import datetime

## 2. Configuration

In [ ]:
# MySQL Configuration
MYSQL_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456'
}

DATABASE_NAME = 'portfolio_backtesting'

print("✅ Configuration loaded")
print(f"   Host: {MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}")
print(f"   Database: {DATABASE_NAME}")

## 3. Connect to MySQL

In [ ]:
# Connect to MySQL server
try:
    connection = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = connection.cursor(dictionary=True)
    print(f"✅ Connected to MySQL server at {MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}")
except Error as e:
    print(f"❌ Error connecting to MySQL: {e}")
    raise

## 4. Create Database

In [ ]:
# Drop existing database if exists
cursor.execute(f"DROP DATABASE IF EXISTS {DATABASE_NAME}")
print(f"🗑️  Dropped existing database '{DATABASE_NAME}' (if existed)")

# Create new database
cursor.execute(f"CREATE DATABASE {DATABASE_NAME} CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci")
print(f"✅ Created database '{DATABASE_NAME}'")

# Use the database
cursor.execute(f"USE {DATABASE_NAME}")
print(f"✅ Using database '{DATABASE_NAME}'")

## 5. Create Tables from Schema

In [ ]:
# Read schema file
with open('database/schema.sql', 'r', encoding='utf-8') as f:
    schema_content = f.read()

# Split into individual statements
statements = []
current = []

for line in schema_content.split('\n'):
    line = line.strip()
    if line.startswith('--') or line.startswith('#') or not line:
        continue
    if line.upper().startswith('CREATE DATABASE') or line.upper().startswith('USE ') or line.upper().startswith('DROP DATABASE'):
        continue
    
    current.append(line)
    if line.rstrip().endswith(';'):
        statements.append(' '.join(current))
        current = []

# Execute each statement
print("📋 Creating tables...")
for stmt in statements:
    if stmt.strip():
        try:
            cursor.execute(stmt)
        except Error as e:
            print(f"⚠️  Error: {e}")

connection.commit()
print(f"✅ Created tables successfully")

# Show created tables
cursor.execute("SHOW TABLES")
tables = [row[f'Tables_in_{DATABASE_NAME}'] for row in cursor.fetchall()]
print(f"\n📊 Tables created ({len(tables)}):")
for table in tables:
    print(f"   - {table}")

## 6. Import ETF Master Data

In [ ]:
print("📥 Importing ETF master data...")

# Read ETF list
etf_df = pd.read_csv('data/etf_list.csv')
print(f"   Found {len(etf_df)} ETFs in file")

# Import each ETF
count = 0
for _, row in etf_df.iterrows():
    try:
        cursor.execute("""
            INSERT INTO etf_master
            (ticker_symbol, etf_name, asset_class, region, sector, expense_ratio, inception_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            row['ticker_symbol'],
            row['etf_name'],
            row['asset_class'],
            row.get('region'),
            row.get('sector'),
            float(row['expense_ratio']) if pd.notna(row.get('expense_ratio')) else None,
            row.get('inception_date') if pd.notna(row.get('inception_date')) else None
        ))
        count += 1
    except Error as e:
        print(f"⚠️  Error importing {row['ticker_symbol']}: {e}")

connection.commit()
print(f"✅ Imported {count} ETFs")

# Verify
cursor.execute("SELECT COUNT(*) as count FROM etf_master")
total = cursor.fetchone()['count']
print(f"   Total in database: {total}")

## 7. Import Benchmark Portfolios

In [ ]:
print("📥 Importing benchmark portfolios...")

# Read benchmarks
benchmark_df = pd.read_csv('data/benchmark_portfolios.csv')
print(f"   Found {len(benchmark_df)} benchmarks in file")

# Import each benchmark
count = 0
for _, row in benchmark_df.iterrows():
    try:
        cursor.execute("""
            INSERT INTO benchmark_portfolios
            (benchmark_name, description, risk_level, target_return, asset_allocation)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            row['benchmark_name'],
            row.get('description'),
            row.get('risk_level', 'Moderate'),
            float(row['target_return']) if pd.notna(row.get('target_return')) else None,
            row.get('asset_allocation')
        ))
        count += 1
    except Error as e:
        print(f"⚠️  Error importing {row['benchmark_name']}: {e}")

connection.commit()
print(f"✅ Imported {count} benchmark portfolios")

# Verify
cursor.execute("SELECT COUNT(*) as count FROM benchmark_portfolios")
total = cursor.fetchone()['count']
print(f"   Total in database: {total}")

## 8. Import Benchmark Holdings

In [ ]:
print("📥 Importing benchmark holdings...")

# Get benchmark and ETF mappings
cursor.execute("SELECT benchmark_id, benchmark_name FROM benchmark_portfolios")
benchmark_map = {row['benchmark_name']: row['benchmark_id'] for row in cursor.fetchall()}

cursor.execute("SELECT etf_id, ticker_symbol FROM etf_master")
etf_map = {row['ticker_symbol']: row['etf_id'] for row in cursor.fetchall()}

# Read holdings
holdings_df = pd.read_csv('data/benchmark_holdings.csv')
print(f"   Found {len(holdings_df)} holdings in file")

# Import each holding
count = 0
for _, row in holdings_df.iterrows():
    benchmark_id = benchmark_map.get(row['benchmark_name'])
    etf_id = etf_map.get(row['ticker_symbol'])
    
    if not benchmark_id or not etf_id:
        continue
    
    try:
        cursor.execute("""
            INSERT INTO benchmark_holdings
            (benchmark_id, etf_id, target_weight)
            VALUES (%s, %s, %s)
        """, (benchmark_id, etf_id, float(row['target_weight'])))
        count += 1
    except Error as e:
        print(f"⚠️  Error importing holding: {e}")

connection.commit()
print(f"✅ Imported {count} benchmark holdings")

# Verify
cursor.execute("SELECT COUNT(*) as count FROM benchmark_holdings")
total = cursor.fetchone()['count']
print(f"   Total in database: {total}")

## 9. Import Price History (This may take 1-2 minutes)

In [ ]:
print("📥 Importing price history (this may take a while)...")

# Check if file exists
if not os.path.exists('data/etf_price_history.csv'):
    print("❌ Price history file not found: data/etf_price_history.csv")
    print("ℹ️  Run 'scripts/generate_sample_data.py' first to generate the data")
else:
    # Get ETF mapping
    cursor.execute("SELECT etf_id, ticker_symbol FROM etf_master")
    etf_map = {row['ticker_symbol']: row['etf_id'] for row in cursor.fetchall()}
    
    # Read price history in chunks for better performance
    chunk_size = 10000
    total_imported = 0
    
    for chunk in pd.read_csv('data/etf_price_history.csv', chunksize=chunk_size):
        # Map ticker to etf_id
        chunk['etf_id'] = chunk['ticker_symbol'].map(etf_map)
        
        # Remove rows without valid etf_id
        chunk = chunk.dropna(subset=['etf_id'])
        chunk['etf_id'] = chunk['etf_id'].astype(int)
        
        # Prepare batch insert
        batch = []
        for _, row in chunk.iterrows():
            batch.append((
                int(row['etf_id']),
                row['date'],
                float(row['open']),
                float(row['high']),
                float(row['low']),
                float(row['close']),
                float(row['adj_close']),
                int(float(row['volume']))
            ))
        
        # Batch insert
        if batch:
            cursor.executemany("""
                INSERT INTO price_history
                (etf_id, date, open, high, low, close, adj_close, volume)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            """, batch)
            connection.commit()
            total_imported += len(batch)
            print(f"   Imported {total_imported:,} records...", end='\r')
    
    print(f"\n✅ Imported {total_imported:,} price records")
    
    # Verify
    cursor.execute("SELECT COUNT(*) as count FROM price_history")
    total = cursor.fetchone()['count']
    print(f"   Total in database: {total:,}")

## 10. Summary - Database Setup Complete!

In [ ]:
print("=" * 70)
print("DATABASE SETUP COMPLETE!".center(70))
print("=" * 70)

# Get all tables and counts
cursor.execute("SHOW TABLES")
tables = [row[f'Tables_in_{DATABASE_NAME}'] for row in cursor.fetchall()]

print(f"\n📊 Database: {DATABASE_NAME}")
print(f"📋 Total Tables: {len(tables)}\n")

for table in tables:
    cursor.execute(f"SELECT COUNT(*) as count FROM {table}")
    count = cursor.fetchone()['count']
    print(f"   {table:30s} {count:>10,} rows")

print("\n✅ Ready to run: main.ipynb")
print("\nDatabase connection will remain open for further use.")

## Close Connection (Optional)

In [ ]:
# Uncomment to close connection
# cursor.close()
# connection.close()
# print("👋 Database connection closed")